In [1]:
import pandas as pd
import requests

def get_precipitation(start_dt , end_dt , lon , lat):

    url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?start={start_dt}&end={end_dt}&latitude={lat}&longitude={lon}&community=ag&parameters=PRECTOTCORR&format=json&header=true")


    response = requests.get(url , timeout = 20)

    response.raise_for_status()

    data = response.json()
    print(data)
    
    coordinates = data['geometry']['coordinates']    

    coordinate_df = pd.DataFrame([coordinates ], columns = ['Latitude' , 'Longitude' , 'Elevation(in m)'])
    
    df = data['properties']['parameter']['PRECTOTCORR']
    precipation_df = pd.DataFrame( list(df.items()),
                                   columns = ["date", "precipitation"])

    return precipation_df , coordinate_df

get_precipitation( 19810101 , 20251230 ,20.593684 ,79 )

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [20.594, 79.0, 209.58]}, 'properties': {'parameter': {'PRECTOTCORR': {'19810101': 1.67, '19810102': 0.55, '19810103': 0.92, '19810104': 0.44, '19810105': 0.35, '19810106': 0.43, '19810107': 0.63, '19810108': 3.78, '19810109': 5.04, '19810110': 1.06, '19810111': 0.64, '19810112': 0.26, '19810113': 0.65, '19810114': 1.1, '19810115': 0.59, '19810116': 0.83, '19810117': 0.9, '19810118': 0.89, '19810119': 0.39, '19810120': 3.39, '19810121': 1.44, '19810122': 2.9, '19810123': 8.79, '19810124': 1.04, '19810125': 0.83, '19810126': 1.64, '19810127': 1.85, '19810128': 0.65, '19810129': 0.43, '19810130': 0.43, '19810131': 1.46, '19810201': 9.9, '19810202': 0.85, '19810203': 3.41, '19810204': 0.79, '19810205': 0.67, '19810206': 3.01, '19810207': 8.46, '19810208': 12.91, '19810209': 7.97, '19810210': 5.5, '19810211': 0.46, '19810212': 0.53, '19810213': 1.2, '19810214': 1.51, '19810215': 0.58, '19810216': 5.01, '19810217': 1.57, '19810

(           date  precipitation
 0      19810101           1.67
 1      19810102           0.55
 2      19810103           0.92
 3      19810104           0.44
 4      19810105           0.35
 ...         ...            ...
 16430  20251226           0.42
 16431  20251227           0.94
 16432  20251228           2.03
 16433  20251229           0.88
 16434  20251230           1.09
 
 [16435 rows x 2 columns],
    Latitude  Longitude  Elevation(in m)
 0    20.594       79.0           209.58)

In [2]:
precipation_df , coordinates = get_precipitation( 19810101 , 20251230 ,20.593684 ,79 )

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [20.594, 79.0, 209.58]}, 'properties': {'parameter': {'PRECTOTCORR': {'19810101': 1.67, '19810102': 0.55, '19810103': 0.92, '19810104': 0.44, '19810105': 0.35, '19810106': 0.43, '19810107': 0.63, '19810108': 3.78, '19810109': 5.04, '19810110': 1.06, '19810111': 0.64, '19810112': 0.26, '19810113': 0.65, '19810114': 1.1, '19810115': 0.59, '19810116': 0.83, '19810117': 0.9, '19810118': 0.89, '19810119': 0.39, '19810120': 3.39, '19810121': 1.44, '19810122': 2.9, '19810123': 8.79, '19810124': 1.04, '19810125': 0.83, '19810126': 1.64, '19810127': 1.85, '19810128': 0.65, '19810129': 0.43, '19810130': 0.43, '19810131': 1.46, '19810201': 9.9, '19810202': 0.85, '19810203': 3.41, '19810204': 0.79, '19810205': 0.67, '19810206': 3.01, '19810207': 8.46, '19810208': 12.91, '19810209': 7.97, '19810210': 5.5, '19810211': 0.46, '19810212': 0.53, '19810213': 1.2, '19810214': 1.51, '19810215': 0.58, '19810216': 5.01, '19810217': 1.57, '19810

In [49]:
import requests 
import pandas as pd

def renew_eng_consum( ):

# | Indicator Code           | Description                            | Unit |
# |
# | ``EN.GHG.CO2.MT.CE.AR5`` | Total CO₂ emissions (excluding LULUCF) | Mt CO₂e |
# | ``EN.GHG.CO2.PC.CE.AR5`` | CO₂ emissions per capita (excluding LULUCF) | t CO₂e per person |

    # url = (f'https://api.worldbank.org/v2/country/ALL/indicator/EG.FEC.RNEW.ZS?format=json&per_page=500')
    meta_url = (f'https://api.worldbank.org/v2/indicator/EG.FEC.RNEW.ZS?format=json')

    # since this api retun data in multiple pages so loopin through pages
    all_data = []
    page = 1
   

    print('Processing request...')
    while True:
        url = (f'https://api.worldbank.org/v2/country/ALL/indicator/EG.FEC.RNEW.ZS?format=json&per_page=2000&page={page}')
        r = requests.get(url , timeout = 20).json()
        
        total_pages = r[0]['pages']

        print(f"\rFetching page {page}/{total_pages}...", end='', flush=True)
        all_data.extend(r[1])
        if page >= r[0]['pages']:
            break
        else:
            page += 1
    print(f'\nDone. Collected {len(all_data)} records .')


     
    meta_response = requests.get(meta_url)
    metadata = meta_response.json()[1][0]

 # converting data into DataFrame
    print('converting into DataFrame')
    df = pd.DataFrame(all_data)        
    df = df[['date' , 'value' , 'countryiso3code' ,'country'  ]]
    df['country'] = df['country'].apply(lambda x: x.get('value') if isinstance(x, dict) else x)

# Renaming columns
    df = df.rename(columns = {'countryiso3code' : 'Country_code'})
    df = df.rename(columns = {'date' : 'year'})

# changing data types of columnms
    df['year'] = df['year'].astype(int)
    df['value'] = df['value'].astype(float)

    print('Got dataFrame shape :' , df.shape)

    return df ,metadata
 

In [50]:
df , metadata = renew_eng_consum( )

Processing request...
Fetching page 9/9...
Done. Collected 17490 records .
converting into DataFrame
Got dataFrame shape : (17490, 4)


In [51]:
metadata
df = df.dropna()

In [52]:
df.head()


,year,value,Country_code,country
5,2020,65.782380,AFE,Africa Eastern and Southern
6,2019,62.690710,AFE,Africa Eastern and Southern
7,2018,61.587530,AFE,Africa Eastern and Southern
8,2017,61.426950,AFE,Africa Eastern and Southern
9,2016,61.822884,AFE,Africa Eastern and Southern


In [53]:
print(df["Country_code"].nunique())
print("India present:", "IND" in df["Country_code"].unique())
df


256
India present: True


,year,value,Country_code,country
5,2020,65.782380,AFE,Africa Eastern and Southern
6,2019,62.690710,AFE,Africa Eastern and Southern
7,2018,61.587530,AFE,Africa Eastern and Southern
8,2017,61.426950,AFE,Africa Eastern and Southern
9,2016,61.822884,AFE,Africa Eastern and Southern
...,...,...,...,...
17455,1994,68.500000,ZWE,Zimbabwe
17456,1993,64.400000,ZWE,Zimbabwe
17457,1992,64.400000,ZWE,Zimbabwe
17458,1991,63.700000,ZWE,Zimbabwe
